# Syntax-LCM data

In [ ]:
import csv
import json

data = {
    'id': [],
    'text': [],
    'score': []
}
with open('study1.csv', newline='') as csvfile:
  reader = csv.DictReader(csvfile)
  for row in reader:
    data['id'].append(row['PARTICIPANT NUMBER'])
    data['text'].append(row['TEXT'])
    if row['Score'] and row['Score']!= ' ':
      data['score'].append(float(row['Score']))
    else:
      data['score'].append(0)
file_name = "study1_data.json"
with open(file_name, 'w') as json_file:
    json.dump(data, json_file, indent=4)

In [ ]:
import csv
import json

data = {
    'id': [],
    'text': [],
    'score': [],
}
test_data = {
    'id': [],
    'text': [],
    'score': [],
    'label': []
}
with open('syntax_data.csv', encoding='latin1', newline='') as csvfile:
  reader = csv.DictReader(csvfile)
  for row in reader:
    if row['Category'] == 'FALSE':
      data['id'].append(len(data['id']))
      data['text'].append(row['Text'])
      data['score'].append(float(row['Score']))
    elif row['Category'] == '1':
      test_data['id'].append(len(test_data['id']))
      test_data['text'].append(row['Text'])
      test_data['score'].append(float(row['Score']))
      test_data['label'].append(1)
    elif row['Category'] == '0':
      test_data['id'].append(len(test_data['id']))
      test_data['text'].append(row['Text'])
      test_data['score'].append(float(row['Score']))
      test_data['label'].append(0)
file_name = "syn_data.json"
with open(file_name, 'w') as json_file:
    json.dump(data, json_file, indent=4)
file_name = "test_data.json"
with open(file_name, 'w') as json_file:
    json.dump(test_data, json_file, indent=4)

# Train model

In [ ]:
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaModel, AdamW
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
class RobertaRegressor(nn.Module):
    def __init__(self, dropout_rate=0.1):
        super(RobertaRegressor, self).__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(dropout_rate)

        # Separate branches for binary and regression
        self.binary_classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 1)
        )

        self.regressor = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)

        # Binary prediction (0 or non-zero)
        binary_logits = self.binary_classifier(pooled_output)
        binary_pred = torch.sigmoid(binary_logits)

        # Regression prediction (1-4 range)
        reg_pred = 1 + 3 * torch.sigmoid(self.regressor(pooled_output))

        # Combine predictions
        final_pred = binary_pred * reg_pred

        return final_pred

In [ ]:
class TextRegressionDataset(Dataset):
  def __init__(self, texts, scores, label=None, tokenizer=None, max_length=128):
    self.texts = texts
    self.scores = scores
    self.label = label if label else [2]*len(self.texts)
    self.tokenizer = tokenizer
    self.max_length = max_length

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, idx):
    text = str(self.texts[idx])
    score = float(self.scores[idx])
    label = int(self.label[idx])

    encoding = self.tokenizer(
        text,
        max_length=self.max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    return {
        'input_ids': encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'score': torch.tensor(score, dtype=torch.float),
        'label': torch.tensor(label, dtype=torch.long)
    }

In [ ]:
class CustomLoss(nn.Module):
    def __init__(self, binary_weight=1.0, regression_weight=1.0):
        super(CustomLoss, self).__init__()
        self.binary_weight = binary_weight
        self.regression_weight = regression_weight
        self.bce = nn.BCELoss()
        self.mse = nn.MSELoss()

    def forward(self, predictions, targets):
        # Binary targets (0 or 1)
        binary_targets = (targets > 0).float()

        # Calculate binary loss
        binary_predictions = (predictions > 0).float()
        binary_loss = self.bce(binary_predictions, binary_targets)

        # Calculate regression loss only for non-zero targets
        mask = targets > 0
        if mask.any():
            regression_loss = self.mse(predictions[mask], targets[mask])
        else:
            regression_loss = 0.0

        # Combine losses
        total_loss = self.binary_weight * binary_loss + self.regression_weight * regression_loss

        return total_loss

In [ ]:
def train_model(model, train_loader, val_loader, device, num_epochs=20):
    optimizer = AdamW(model.parameters(), lr=1e-5)  # Reduced learning rate
    criterion = CustomLoss(binary_weight=1.0, regression_weight=1.0)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2)
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs.squeeze(), scores)
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0
        all_predictions = []
        all_targets = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                scores = batch['score'].to(device)

                outputs = model(input_ids, attention_mask)
                val_loss += criterion(outputs.squeeze(), scores).item()

                all_predictions.extend(outputs.squeeze().cpu().numpy())
                all_targets.extend(scores.cpu().numpy())

        avg_train_loss = total_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        # Print detailed statistics
        predictions = np.array(all_predictions)
        targets = np.array(all_targets)
        print(f'\nEpoch {epoch + 1}:')
        print(f'Training Loss: {avg_train_loss:.4f}')
        print(f'Validation Loss: {avg_val_loss:.4f}')
        print(f'Predictions stats: min={predictions.min():.2f}, max={predictions.max():.2f}, mean={predictions.mean():.2f}')
        print(f'Number of non-zero predictions: {(predictions > 0).sum()}')

        # Adjust learning rate
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pt')
            print('Saved best model!')
        print('-' * 50)

In [ ]:
def evaluate_model(model, test_loader, device, threshold=3):
  model.eval()
  all_predictions = []
  all_true_scores = []
  binary_true = []

  with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):

      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      scores = batch['score'].to(device)

      outputs = model(input_ids, attention_mask)
      predictions = outputs.squeeze().cpu().numpy()
      true_scores = scores.cpu().numpy()

      all_predictions.extend(predictions)
      all_true_scores.extend(true_scores)
      binary_true.extend(batch['label'])

  all_predictions = np.array(all_predictions)
  all_true_scores = np.array(all_true_scores)

  # Calculate metrics
  rmse = np.sqrt(mean_squared_error(all_true_scores, all_predictions))
  mae = mean_absolute_error(all_true_scores, all_predictions)
  r2 = r2_score(all_true_scores, all_predictions)

  # Calculate metrics for zero vs non-zero predictions
  zero_mask = all_true_scores == 0
  nonzero_mask = all_true_scores != 0

  # Changed to: Convert predictions to binary (concrete vs abstract)
  binary_predictions = (1 - (all_true_scores < 3)).astype(float)
  accuracy = np.mean(binary_predictions == binary_true)

  # Calculate error metrics for non-zero values only
  if np.any(nonzero_mask):
    nonzero_rmse = np.sqrt(mean_squared_error(
        all_true_scores[nonzero_mask],
        all_predictions[nonzero_mask]
    ))
    nonzero_mae = mean_absolute_error(
        all_true_scores[nonzero_mask],
        all_predictions[nonzero_mask]
    )
  else:
    nonzero_rmse = nonzero_mae = float('nan')

  metrics = {
    'overall_rmse': rmse,
    'overall_mae': mae,
    'r2_score': r2,
    'binary_accuracy': accuracy,
    'nonzero_rmse': nonzero_rmse,
    'nonzero_mae': nonzero_mae,
    'num_zero_samples': np.sum(zero_mask),
    'num_nonzero_samples': np.sum(nonzero_mask)
  }

  return metrics, all_predictions, all_true_scores, binary_predictions, binary_true

In [ ]:
import random

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('study1_data.json', 'r', encoding='utf-8') as f:
  train_data = json.load(f)
with open('syn_data.json', 'r', encoding='utf-8') as f:
  syn1_data = json.load(f)
with open('test_data.json', 'r', encoding='utf-8') as f:
  syn2_data = json.load(f)


train_texts = train_data['text']
train_scores = train_data['score']

combined_data_len = len(syn1_data['text'])+len(syn2_data['text'])
num_indices = int(0.5 * combined_data_len)
random_indices = random.sample(range(combined_data_len), num_indices)
syn_texts = syn1_data['text']+syn2_data['text']
syn_scores = syn1_data['score']+syn2_data['score']
val_texts = [syn_texts[i] for i in random_indices]
val_scores = [syn_scores[i] for i in random_indices]
test_texts = [syn_texts[i] for i in range(combined_data_len) if i not in random_indices]
test_scores = [syn_scores[i] for i in range(combined_data_len) if i not in random_indices]
test_labels = [1 for i in range(combined_data_len) if i not in random_indices]

# Create datasets
train_dataset = TextRegressionDataset(train_texts, train_scores, None, tokenizer)
val_dataset = TextRegressionDataset(val_texts, val_scores, None, tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Initialize and train model
model = RobertaRegressor().to(device)

# Training
print("Starting training...")
train_model(model, train_loader, val_loader, device)



In [ ]:
test_dataset = TextRegressionDataset(test_texts, test_scores, test_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16)

# Load best model for testing
print("\nLoading best model for testing...")
model.load_state_dict(torch.load('best_model.pt'))

# Test the model
print("\nEvaluating model on test set...")
metrics, predictions, true_scores, binary_predictions, binary_true = evaluate_model(model, test_loader, device)

# Print test results
print("\nTest Set Metrics:")
print(f"Overall RMSE: {metrics['overall_rmse']:.4f}")
print(f"Overall MAE: {metrics['overall_mae']:.4f}")
print(f"R² Score: {metrics['r2_score']:.4f}")
print(f"Binary Classification Accuracy: {metrics['binary_accuracy']:.4f}")
print(f"RMSE for non-zero scores: {metrics['nonzero_rmse']:.4f}")
print(f"MAE for non-zero scores: {metrics['nonzero_mae']:.4f}")
print(f"Number of zero samples: {metrics['num_zero_samples']}")
print(f"Number of non-zero samples: {metrics['num_nonzero_samples']}")

# Optional: Save predictions for further analysis
results = {
    'predictions': predictions.tolist(),
    'true_scores': true_scores.tolist(),
    'binary_predictions': binary_predictions.tolist(),
    'binary_true': [int(tensor.item()) for tensor in binary_true],
    # 'metrics': metrics
}
with open('test_results.json', 'w') as f:
    json.dump(results, f)

print("\nTest results saved to 'test_results.json'")

# Inference - ABSA

In [ ]:
import pickle

# Specify the file path
file_path = 'ABSA_Openreview/iclr_conf.p'

# Load the pickle file
with open(file_path, 'rb') as file:
    data = pickle.load(file)

In [ ]:
new_data = {
    'id':[],
    'year': [],
    'decision': [],
    'comment': [],
    'confidence': [],
    'rating': [],
}
for d in data:
  paper_id = d[0].split('forum=')[-1]
  decision = d[1]
  year = d[2]
  count = 0
  for c in d[3]:
    comment = c[0]
    confidence = c[1]
    rating = c[2]
    new_data['id'].append(f"{paper_id}_{count}")
    new_data['decision'].append(decision)
    new_data['year'].append(year)
    new_data['comment'].append(comment)
    new_data['confidence'].append(confidence)
    new_data['rating'].append(rating)
    count += 1

In [ ]:
import json
with open('openreview_data.json', 'w') as f:
  json.dump(new_data, f, indent=2)

In [ ]:
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaModel
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import json

In [ ]:
class SentenceLevelDataset(Dataset):
    def __init__(self, sentences_list, data_ids, tokenizer, max_length=512):
        # Flatten the structure while keeping track of original IDs
        self.sentences = []
        self.data_ids = []
        self.sentence_indices = []  # Track sentence position within each sample

        for sample_idx, (sentences, data_id) in enumerate(zip(sentences_list, data_ids)):
            for sent_idx, sentence in enumerate(sentences):
                self.sentences.append(sentence)
                self.data_ids.append(data_id)
                self.sentence_indices.append(sent_idx)

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        # Tokenize single sentence
        encoding = self.tokenizer(
            self.sentences[idx],
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'data_id': self.data_ids[idx],
            'sentence_idx': self.sentence_indices[idx],
            # 'original_sentence': self.sentences[idx]
        }

In [ ]:
class SentenceDataset(Dataset):
    def __init__(self, sentences_list, data_ids, tokenizer, max_length=512):
        self.sentences = sentences_list
        self.data_ids = data_ids
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        # Join multiple sentences with space
        text = " ".join(self.sentences[idx])

        # Tokenize the text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'data_id': self.data_ids[idx]
        }

In [ ]:
class RobertaRegressor(nn.Module):
    def __init__(self, dropout_rate=0.1):
        super(RobertaRegressor, self).__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(dropout_rate)

        # Separate branches for binary and regression
        self.binary_classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 1)
        )

        self.regressor = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)

        # Binary prediction (0 or non-zero)
        binary_logits = self.binary_classifier(pooled_output)
        binary_pred = torch.sigmoid(binary_logits)

        # Regression prediction (1-4 range)
        reg_pred = 1 + 3 * torch.sigmoid(self.regressor(pooled_output))

        # Combine predictions
        final_pred = binary_pred * reg_pred

        return final_pred

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RobertaRegressor().to(device)
model.load_state_dict(torch.load('best_model.pt'))

In [ ]:
texts = new_data['comment']
ids = new_data['id']

dataset = SentenceLevelDataset(texts, ids, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

results = []
all_predictions = []
all_ids = []

# Inference loop
with torch.no_grad():
  for batch in tqdm(dataloader, desc="Making predictions"):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)

    outputs = model(input_ids, attention_mask=attention_mask)
    predictions = outputs.squeeze().cpu().numpy()

    # Handle both single and batch predictions
    if predictions.ndim == 0:
        predictions = [predictions.item()]
    else:
        predictions = predictions.tolist()

    all_predictions.extend(predictions)
    all_ids.extend(batch['data_id'])


In [ ]:
texts = new_data['comment']
ids = new_data['id']

dataset = SentenceDataset(texts, ids, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

# Lists to store predictions and corresponding IDs
all_predictions = []
all_ids = []

# Inference loop
with torch.no_grad():
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        data_ids = batch['data_id']

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = outputs.squeeze().cpu().numpy()

        all_predictions.extend(predictions.tolist())
        all_ids.extend(data_ids)

In [ ]:
from collections import defaultdict

results_dict = defaultdict(list)
for id_, pred in zip(all_ids, all_predictions):
    results_dict[id_].append(pred)

# Convert defaultdict to regular dict
results_dict = dict(results_dict)
with open('raw_predictions_p.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

for key, value in results_dict.items():
    results_dict[key] = np.mean(value)

In [ ]:
 # Add predictions to original data
new_data['predicted_score'] = ['']*len(new_data['id'])
for idx, id in enumerate(new_data['id']):
    score = results_dict[id]
    new_data['predicted_score'][idx] = score

# Save merged results
with open('merged_predictions_p.json', 'w') as f:
    json.dump(new_data, f, indent=2)